In [2]:
# finetune_donut_invoice.py
import os
import re
import json
from dataclasses import dataclass
from typing import Any, Dict, List

import torch
from datasets import load_dataset, Dataset
from transformers import (
    DonutProcessor,
    VisionEncoderDecoderModel,
    TrainingArguments,
    Trainer,
    default_data_collator,
)
import evaluate

# ------------- CONFIG -------------
PRETRAINED = "naver-clova-ix/donut-base"  # pre-trained Donut base
TASK_TOKEN = "<s_invoice>"
END_TASK_TOKEN = "</s_invoice>"
IMAGE_SIZE = (896, 896)  # bisa disesuaikan (H, W)
BATCH_SIZE = 4
LR = 5e-5
NUM_EPOCHS = 5
OUTPUT_DIR = "./donut-invoice-finetuned"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# ----------------------------------

# ------------- utils -------------
def load_local_jsonlines(path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            data.append(json.loads(line))
    return data

def wrap_target_json(json_str):
    # remove whitespace between keys to keep tokenization stable (optional)
    # keep as valid JSON string
    s = json_str.strip()
    return f"{TASK_TOKEN}{s}{END_TASK_TOKEN}"

# normalize json strings for EM metric
def normalize_json_str(s: str):
    # remove task tokens and whitespace between keys/values (simple)
    s = re.sub(r"<.*?>", "", s)
    try:
        # parse & dump with sorted keys for canonical form
        obj = json.loads(s)
        return json.dumps(obj, sort_keys=True, ensure_ascii=False)
    except Exception:
        # fallback: strip whitespace
        return re.sub(r"\s+", "", s).strip()
# ----------------------------------

# ------------- Load processor & model -------------
processor = DonutProcessor.from_pretrained(PRETRAINED)
model = VisionEncoderDecoderModel.from_pretrained(PRETRAINED)
model.to(DEVICE)

# Add task tokens to tokenizer if not present
tokenizer = processor.tokenizer
additional = []
if TASK_TOKEN not in tokenizer.get_vocab():
    additional.append(TASK_TOKEN)
if END_TASK_TOKEN not in tokenizer.get_vocab():
    additional.append(END_TASK_TOKEN)
if additional:
    tokenizer.add_tokens(additional, special_tokens=True)
    model.decoder.resize_token_embeddings(len(tokenizer))

# adjust model config if needed
model.config.decoder_start_token_id = tokenizer.cls_token_id or tokenizer.bos_token_id
model.config.pad_token_id = tokenizer.pad_token_id
model.config.eos_token_id = tokenizer.eos_token_id
# optional: limit generation length
model.config.decoder.max_length = 512
# ----------------------------------

# ------------- Prepare dataset -------------
# Example: assume you have a jsonl with fields image_path and label (json string)
# Replace with your dataset path
TRAIN_JSONL = "data/invoices_train.jsonl"
VAL_JSONL = "data/invoices_val.jsonl"

def prepare_dataset_from_jsonl(jsonl_path):
    rows = load_local_jsonlines(jsonl_path)
    # convert to Hugging Face dataset
    ds = Dataset.from_list(rows)
    return ds

# load datasets
train_ds = prepare_dataset_from_jsonl(TRAIN_JSONL)
val_ds = prepare_dataset_from_jsonl(VAL_JSONL)

# preprocessing function
def preprocess_examples(examples):
    images = []
    for p in examples["image_path"]:
        img = processor.image_processor.load_image(p).convert("RGB")
        images.append(img)

    # pixel_values: shape (batch, channels, H, W)
    pixel_values = processor(images=images, return_tensors="pt").pixel_values

    # prepare labels (tokenize target string)
    targets = [wrap_target_json(t) for t in examples["label"]]

    tokenized_targets = tokenizer(
        targets,
        add_special_tokens=False,  # Donut uses its own special tokens
        padding="longest",
        truncation=True,
        max_length=model.config.decoder.max_length,
        return_tensors="pt",
    )

    # return dicts suitable for Trainer
    batch = {
        "pixel_values": pixel_values,
        "labels": tokenized_targets.input_ids,
    }
    # attention mask for decoder labels (optional; Trainer will handle)
    return batch

# map with batched processing
train_ds = train_ds.map(
    lambda batch: preprocess_examples(batch),
    batched=True,
    remove_columns=train_ds.column_names,
)
val_ds = val_ds.map(
    lambda batch: preprocess_examples(batch),
    batched=True,
    remove_columns=val_ds.column_names,
)

# Convert to torch format required by Trainer (set format)
train_ds.set_format(type="torch")
val_ds.set_format(type="torch")
# ----------------------------------

# ------------- Metrics -------------
# We'll use a simple exact match on normalized JSON
def compute_metrics(eval_preds):
    preds_ids, labels_ids = eval_preds
    # decode predictions
    preds = processor.batch_decode(preds_ids, skip_special_tokens=False)
    preds = [p.replace(processor.tokenizer.eos_token, "").replace(processor.tokenizer.pad_token, "") for p in preds]
    preds_norm = [normalize_json_str(p) for p in preds]

    # decode labels (labels_ids may include -100; replace with pad_token_id before decode)
    labels_ids = [[(l if l != -100 else tokenizer.pad_token_id) for l in lab] for lab in labels_ids]
    labels = tokenizer.batch_decode(labels_ids, skip_special_tokens=False)
    labels_norm = [normalize_json_str(l) for l in labels]

    # compute exact match
    em = sum(1 for p, g in zip(preds_norm, labels_norm) if p == g) / len(preds_norm)
    return {"exact_match": em}

# Use evaluate library compatibility (optional)
# ----------------------------------

# ------------- TrainingArguments & Trainer -------------
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    evaluation_strategy="steps",
    eval_steps=200,
    save_steps=500,
    logging_steps=50,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LR,
    fp16=torch.cuda.is_available(),
    save_total_limit=3,
    remove_unused_columns=False,  # important for VisionEncoderDecoderModel
    push_to_hub=False,
)

data_collator = default_data_collator

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    tokenizer=processor.feature_extractor,  # pass feature_extractor for image processing compatibility
    compute_metrics=compute_metrics,
)

# ------------- Run training -------------
if __name__ == "__main__":
    trainer.train()
    trainer.save_model(OUTPUT_DIR)


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


FileNotFoundError: [Errno 2] No such file or directory: 'data/invoices_train.jsonl'